In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("ChurnPipeline")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.driver.bindAddress", "127.0.0.1")
         .getOrCreate())
print(spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/15 20:04:59 WARN Utils: Your hostname, Kartavyas-Macbook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.102 instead (on interface en0)
26/10/15 20:04:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/mainframe/Desktop/big_data_project/venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/15 20:04:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/15 20:05:00 

4.2.0


In [2]:
cust = spark.read.parquet("../silver/customers")
tix  = spark.read.parquet("../silver/tickets")

In [3]:
risk = cust.filter((F.col("Contract") == "Month-to-month") & (F.col("tenure") < 12))
print(risk.count())

1908


In [4]:
tix_agg = tix.groupBy("customerID").agg(
    F.count("*").alias("ticket_cnt"),
    F.round(F.avg("resolution_hrs"), 1).alias("avg_resolution_hrs"))

joined = (cust.join(tix_agg, "customerID", "left")
              .fillna({"ticket_cnt": 0, "avg_resolution_hrs": 0.0}))

In [5]:
(joined.groupBy("Contract")
       .agg(F.round(F.avg("churn_flag") * 100, 2).alias("churn_pct"),
            F.count("*").alias("customers"))
       .orderBy(F.desc("churn_pct")).show())

+--------------+---------+---------+
|      Contract|churn_pct|customers|
+--------------+---------+---------+
|Month-to-month|    42.71|     3875|
|      One year|    11.27|     1473|
|      Two year|     2.83|     1695|
+--------------+---------+---------+



In [6]:
joined.createOrReplaceTempView("customers")
spark.sql("""
  SELECT InternetService, PaymentMethod,
         ROUND(AVG(churn_flag) * 100, 2) AS churn_pct, COUNT(*) AS customers
  FROM customers
  GROUP BY InternetService, PaymentMethod
  ORDER BY churn_pct DESC
""").show(truncate=False)

+---------------+-------------------------+---------+---------+
|InternetService|PaymentMethod            |churn_pct|customers|
+---------------+-------------------------+---------+---------+
|Fiber optic    |Electronic check         |53.23    |1595     |
|Fiber optic    |Mailed check             |42.64    |258      |
|DSL            |Electronic check         |31.94    |648      |
|Fiber optic    |Bank transfer (automatic)|28.95    |646      |
|Fiber optic    |Credit card (automatic)  |25.29    |597      |
|DSL            |Mailed check             |20.72    |613      |
|No             |Electronic check         |12.3     |122      |
|DSL            |Credit card (automatic)  |12.12    |594      |
|No             |Mailed check             |9.58     |741      |
|DSL            |Bank transfer (automatic)|9.36     |566      |
|No             |Bank transfer (automatic)|5.42     |332      |
|No             |Credit card (automatic)  |2.72     |331      |
+---------------+-----------------------

26/10/15 20:05:02 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [7]:
w = Window.partitionBy("Contract").orderBy(F.desc("MonthlyCharges"))
joined.withColumn("charge_rank", F.dense_rank().over(w)) \
      .select("customerID", "Contract", "MonthlyCharges", "charge_rank").show(10)

w2 = Window.partitionBy("customerID").orderBy("ticket_date")
(tix.withColumn("ticket_seq", F.row_number().over(w2))
    .withColumn("days_since_prev", F.datediff("ticket_date", F.lag("ticket_date").over(w2)))
    .orderBy("customerID", "ticket_date").show(10))

+----------+--------------+--------------+-----------+
|customerID|      Contract|MonthlyCharges|charge_rank|
+----------+--------------+--------------+-----------+
|2302-ANTDP|Month-to-month|        117.45|          1|
|8016-NCFVO|Month-to-month|         116.5|          2|
|9659-QEQSY|Month-to-month|        115.65|          3|
|4361-BKAXE|Month-to-month|         114.5|          4|
|6710-HSJRD|Month-to-month|         114.1|          5|
|9158-VCTQB|Month-to-month|         113.6|          6|
|7279-BUYWN|Month-to-month|         113.2|          7|
|0771-WLCLA|Month-to-month|        112.95|          8|
|1583-IHQZE|Month-to-month|        112.95|          8|
|9481-IEBZY|Month-to-month|         112.9|          9|
+----------+--------------+--------------+-----------+
only showing top 10 rows
+---------+----------+-----------+--------+--------------+--------------------+--------------------+---------+----------+---------------+
|ticket_id|customerID|ticket_date|category|resolution_hrs|        _

In [8]:
service_cols = ["PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
                "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]

features = (joined
  .withColumn("tenure_bucket",
      F.when(F.col("tenure") < 12, "0-12")
       .when(F.col("tenure") < 24, "12-24")
       .when(F.col("tenure") < 48, "24-48")
       .otherwise("48+"))
  .withColumn("avg_monthly_spend",
      F.when(F.col("tenure") > 0, F.col("TotalCharges") / F.col("tenure"))
       .otherwise(F.col("MonthlyCharges")))          # guards against divide-by-zero
  .withColumn("num_services",
      sum(F.when(F.col(c) == "Yes", 1).otherwise(0) for c in service_cols))
)

features.write.mode("overwrite").parquet("../silver/customer_features")
features.select("customerID", "tenure_bucket", "avg_monthly_spend",
                "num_services", "ticket_cnt", "churn_flag").show(10)

+----------+-------------+------------------+------------+----------+----------+
|customerID|tenure_bucket| avg_monthly_spend|num_services|ticket_cnt|churn_flag|
+----------+-------------+------------------+------------+----------+----------+
|0002-ORFBO|         0-12| 65.92222222222222|           4|         2|         0|
|0003-MKNFE|         0-12|60.266666666666666|           3|         2|         0|
|0004-TLHLJ|         0-12|           70.2125|           2|         4|         1|
|0011-IGKFF|        12-24| 95.21923076923076|           5|         2|         1|
|0013-EXCHZ|         0-12| 89.13333333333333|           3|         3|         1|
|0013-MHZWF|         0-12| 63.49444444444445|           4|         4|         0|
|0013-SMEOE|          48+| 111.3274647887324|           7|         1|         0|
|0014-BMAQU|          48+| 85.36190476190477|           4|         3|         0|
|0015-UOCOJ|         0-12|48.621428571428574|           2|         3|         0|
|0016-QLJIS|          48+|  